# Performing MJO filtering in 4 steps=>

📁 STEP 0: Setup folders

In [1]:
import os

base_dir = r"D:\PHD IIT KGP\6.MJO_FILTER"

folders = [
    "01_raw_combined",
    "02_mm_day",
    "03_region_mean",
    "04_climatology",
    "05_anomaly",
    "06_detrended",
    "07_bandpass",
    "08_standardized",
    "09_rmm",
    "10_composites"
]

for f in folders:
    os.makedirs(os.path.join(base_dir, f), exist_ok=True)

print("Folders created")

Folders created


🌧️ STEP 1: Load & Combine 45 years

In [2]:
import xarray as xr
import glob

files = sorted(glob.glob(r"D:\PHD IIT KGP\5.ERA5_1980-2025_12Months_DailyPrep_unzipped\*.nc"))

ds = xr.open_mfdataset(files, combine='by_coords')

ds.to_netcdf(os.path.join(base_dir, "01_raw_combined", "tp_combined.nc"))

: 

## The above Combination is done, not by this particular code

In [2]:
import xarray as xr

file = r"d:\PHD IIT KGP\6.MJO_FILTER\01_raw_combined\tp_combined.nc"

ds = xr.open_dataset(
    file,
    chunks={"time": 365, "latitude": 100, "longitude": 100}
)

print(ds)

<xarray.Dataset> Size: 27GB
Dimensions:    (time: 16802, latitude: 501, longitude: 801)
Coordinates:
  * time       (time) datetime64[ns] 134kB 1980-01-01 1980-01-02 ... 2025-12-31
  * latitude   (latitude) float64 4kB 50.0 49.9 49.8 49.7 ... 0.3 0.2 0.1 0.0
  * longitude  (longitude) float64 6kB 40.0 40.1 40.2 40.3 ... 119.8 119.9 120.0
    number     int64 8B ...
    expver     (time) <U4 269kB dask.array<chunksize=(365,), meta=np.ndarray>
Data variables:
    tp         (time, latitude, longitude) float32 27GB dask.array<chunksize=(365, 100, 100), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-20T08:46 GRIB to CDM+CF via cfgrib-0.9.1...


📅 STEP 2: Daily Climatology (FULL GRID)

In [ ]:
ds = xr.open_dataset(
    file,
    chunks={"time": -1, "latitude": 100, "longitude": 100}
)

clim = ds['tp'].groupby('time.dayofyear').mean('time')

In [3]:
import xarray as xr
from dask.diagnostics import ProgressBar

# ============================================
# PATHS
# ============================================

input_file  = r"D:\PHD IIT KGP\6.MJO_FILTER\01_raw_combined\tp_combined.nc"
output_file = r"D:\PHD IIT KGP\6.MJO_FILTER\04_climatology\climatology.nc"

# ============================================
# LOAD DATA (SAFE CHUNKING)
# ============================================

ds = xr.open_dataset(
    input_file,
    chunks={
        "time": 20,
        "latitude": 40,
        "longitude": 40
    }
)

# ============================================
# REGION SELECTION
# ============================================

north, west, south, east = 50, 40, 0, 120

# Check latitude order once (important)
if ds.latitude[0] > ds.latitude[-1]:
    ds = ds.sel(latitude=slice(north, south), longitude=slice(west, east))
else:
    ds = ds.sel(latitude=slice(south, north), longitude=slice(west, east))

# ============================================
# MEMORY OPTIMIZATION
# ============================================

tp = ds['tp'].astype('float32')

# ============================================
# CLIMATOLOGY (LAZY)
# ============================================

clim = tp.groupby('time.dayofyear').mean('time', skipna=True)

# ============================================
# PROGRESS BAR + SAVE
# ============================================

print("🚀 Starting computation...")

with ProgressBar():
    clim.to_netcdf(output_file, compute=True)

print("✅ Done! Climatology saved.")

🚀 Starting computation...
[########################################] | 100% Completed | 42m 19s
✅ Done! Climatology saved.


⚡ STEP 3: Anomaly (GRID-WISE)

In [4]:
import xarray as xr
from dask.diagnostics import ProgressBar

# ============================================
# PATHS
# ============================================

input_file   = r"D:\PHD IIT KGP\6.MJO_FILTER\01_raw_combined\tp_combined.nc"
clim_file    = r"D:\PHD IIT KGP\6.MJO_FILTER\04_climatology\climatology.nc"
output_file  = r"D:\PHD IIT KGP\6.MJO_FILTER\05_anomaly\tp_anomaly.nc"

# ============================================
# LOAD DATA (SAFE CHUNKING)
# ============================================

ds = xr.open_dataset(
    input_file,
    chunks={"time": 20, "latitude": 40, "longitude": 40}
)

clim = xr.open_dataset(clim_file)

# ============================================
# REGION SELECTION (same as before!)
# ============================================

north, west, south, east = 50, 40, 0, 120

if ds.latitude[0] > ds.latitude[-1]:
    ds = ds.sel(latitude=slice(north, south), longitude=slice(west, east))
else:
    ds = ds.sel(latitude=slice(south, north), longitude=slice(west, east))

# ============================================
# MEMORY OPTIMIZATION
# ============================================

tp = ds['tp'].astype('float32')

# ============================================
# ANOMALY CALCULATION
# ============================================

anom = tp.groupby('time.dayofyear') - clim['tp']

# ============================================
# SAVE WITH PROGRESS BAR
# ============================================

print("🚀 Computing anomalies...")

with ProgressBar():
    anom.to_netcdf(output_file, compute=True)

print("✅ Anomaly file saved!")

🚀 Computing anomalies...
[                                        ] | 0% Completed | 93m 12ss


MemoryError: 